# **GroupBy, Aggregation, and Transformation**

**DS112 — Computer Programming**

**Week 15:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Demonstrate aggregation tools on toy and real-life datasets.

Many questions compare **groups**: *What is the average grade per section? Which canteen branch sells more? Which weather station is the rainiest?* pandas answers them with **`groupby`**, which follows the **split–apply–combine** strategy.

```{mermaid}
flowchart LR
    D["DataFrame"] --> S1["DS1A rows"]
    D --> S2["DS1B rows"]
    D --> S3["DS1C rows"]
    S1 --> A1["mean"]
    S2 --> A2["mean"]
    S3 --> A3["mean"]
    A1 --> C["combined result<br/>one row per section"]
    A2 --> C
    A3 --> C
```

> **Analogy.** After an exam, the teacher **splits** the answer sheets into piles by section, **applies** the same calculation to each pile (the average), and **combines** the results into one summary table.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

grades = pd.read_csv("data/grades.csv")
sales = pd.read_csv("data/canteen_sales.csv", parse_dates=["date"])
menu = pd.read_csv("data/menu.csv")
weather = pd.read_csv("data/weather.csv", parse_dates=["date"])
sales.head()

## **Grouping and Aggregating**

In [ ]:
grades.groupby("section")["final"].mean().round(2)

In [ ]:
grades.groupby("section")[["quiz", "final"]].agg(["mean", "max"]).round(1)

### **Named aggregation**

Give each result column a clear name with `agg(new_name=(column, function))`:

In [ ]:
(
    grades.groupby("section")
    .agg(students=("name", "count"),
         avg_final=("final", "mean"),
         best_final=("final", "max"),
         missing_midterms=("midterm", lambda s: s.isna().sum()))
    .round(2)
)

| Aggregation | Meaning |
| ----------- | ------- |
| `count` | non-missing values |
| `size` | number of rows (including missing) |
| `sum`, `mean`, `median`, `std` | totals and statistics |
| `min`, `max`, `first`, `last` | extremes and positions |
| `nunique` | number of distinct values |

## **Grouping by Several Columns**

In [ ]:
items_by_branch = sales.groupby(["branch", "item_code"])["qty"].sum()
items_by_branch.head(10)

In [ ]:
items_by_branch.unstack("branch")          # turn the inner group into columns

### **Grouping by time**

In [ ]:
sales["weekday"] = sales["date"].dt.day_name()
daily = sales.groupby(["date", "branch"])["qty"].sum().reset_index()
daily.groupby("branch")["qty"].agg(["mean", "min", "max"]).round(1)

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
sales.groupby("weekday")["qty"].sum().reindex(weekday_order)

## **Transformation**

`agg` **reduces** each group to one value. `transform` returns a result **for every original row**, which is useful for comparing each row with its group.

In [ ]:
grades["section_avg"] = grades.groupby("section")["final"].transform("mean").round(2)
grades["vs_section"] = (grades["final"] - grades["section_avg"]).round(2)
grades[["name", "section", "final", "section_avg", "vs_section"]].head(6)

In [ ]:
# z-score of each reading within its own station
weather["temp_z"] = weather.groupby("station")["temp_c"].transform(lambda s: (s - s.mean()) / s.std()).round(2)
weather.sort_values("temp_z", ascending=False)[["date", "station", "temp_c", "temp_z"]].head(3)

## **Filtering Groups**

`filter` keeps or drops **whole groups** based on a condition:

In [ ]:
big_sections = grades.groupby("section").filter(lambda g: len(g) >= 9)
print(big_sections["section"].value_counts())

## **Putting It Together: Weather Summary**

In [ ]:
weather["month"] = weather["date"].dt.month_name()
(
    weather.groupby(["station", "month"], sort=False)
    .agg(avg_temp=("temp_c", "mean"),
         total_rain=("rain_mm", "sum"),
         rainy_days=("rain_mm", lambda s: (s > 1).sum()))
    .round(1)
)

:::{admonition} Paradigm lens · functional / declarative
:class: tip

`groupby(...).agg(...)` says **what** summary to compute for each group; pandas handles the splitting and looping. Split–apply–combine is the same **map → reduce** pattern from functional programming, applied group by group.
:::

## **Summary**

| Task | Code |
| ---- | ---- |
| one statistic per group | `df.groupby("g")["col"].mean()` |
| several statistics | `.agg(["mean", "max"])` or named `agg(new=(col, func))` |
| several keys | `groupby(["a", "b"])`, then `unstack()` |
| keep row-level detail | `transform("mean")` |
| keep/drop whole groups | `groupby("g").filter(func)` |

## **References**

Maulion, M. R. (2022). *DS 112 computer programming* [Instructional material]. University of Science and Technology of Southern Philippines.

McKinney, W. (2022). *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3rd ed.). O'Reilly Media. https://wesmckinney.com/book/

The pandas development team. (2026). *Group by: split-apply-combine*. https://pandas.pydata.org/docs/user_guide/groupby.html

VanderPlas, J. (2023). *Python data science handbook: Essential tools for working with data* (2nd ed.). O'Reilly Media. https://jakevdp.github.io/PythonDataScienceHandbook/

Wickham, H. (2011). The split-apply-combine strategy for data analysis. *Journal of Statistical Software, 40*(1), 1–29. https://doi.org/10.18637/jss.v040.i01